<a href="https://colab.research.google.com/github/Protein-Function-Prediction/COMP3608ProteinFunctionPrediction/blob/main/dataset1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **COMP 3608 – Protein Function Classification**
## **Dataset 1: Simulated Bioinformatics (5‑class)**

**Notebook purpose:** End‑to‑end pipeline for Dataset 1 – data loading, cleaning, feature engineering, model training (LR, SVM, CNN), tuning, cross‑validation, and evaluation.

## **1) Problem Classification & Algorithm Justification**

This project addresses a **multi‑class classification problem** – predicting one of five functional classes (Enzyme, Structural, Receptor, Transporter, Other) from biophysical descriptors.

**Algorithms selected:**
- **Logistic Regression (LR)** – robust linear baseline, fast, interpretable.
- **Support Vector Machine (SVM) with RBF kernel** – captures non‑linear decision boundaries.
- **1‑D Convolutional Neural Network (CNN)** – learns hierarchical local patterns from the feature vector.

All three are evaluated on the same feature set to ensure a fair comparison.

## **2) Objective Function (Z)**

The primary metric is the **Macro‑F1 score**:

$$F1_k = 2 \times \frac{\text{Precision}_k \times \text{Recall}_k}{\text{Precision}_k + \text{Recall}_k}$$

$$Z = \text{Macro‑F1} = \frac{1}{K} \sum_{k=1}^{K} F1_k$$

This unweighted mean treats all 5 classes equally, preventing majority‑class bias.

## **3) Experimental Design**

- **Stratified 80/20 train‑test split** preserves class proportions.
- **StandardScaler** applied before all models.
- **Class‑weighting:** `class_weight='balanced'` for LR/SVM; class‑weighted CrossEntropyLoss for CNN.
- **Hyperparameter tuning:** `GridSearchCV` for LR/SVM; manual hold‑out grid search for CNN.
- **5‑fold stratified cross‑validation** on training data for robust performance estimates.
- **Macro‑F1** reported as the main metric, accuracy as supplementary.

## **4) Environment Setup & Imports**

In [ ]:
# Install requirements (works on Colab & local Python)
import sys, subprocess, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q -r ../requirements.txt
    %pip install -q kagglehub==0.1.6 kagglesdk==1.5.0
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "../requirements.txt"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub==0.1.6", "kagglesdk==1.5.0"])

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import kagglehub

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline   # SMOTE inside CV (not needed for ds1, but kept for consistency)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

RANDOM_STATE = 42
TEST_SIZE = 0.20
OUTPUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Environment ready. Device:", device)

## **5) Data Download & Loading**

In [ ]:
# Download Dataset 1 from Kaggle
DATA_DIR = Path("data/dataset1")
DATA_DIR.mkdir(parents=True, exist_ok=True)

path_sim1 = kagglehub.dataset_download(
    "willianoliveiragibin/bioinformatics-simulated",
    output_dir=str(DATA_DIR)
)
print("Dataset downloaded to:", path_sim1)

# Load the CSV
csv_path = list(Path(path_sim1).glob("*.csv"))[0]
df_raw = pd.read_csv(csv_path)
print(f"Shape: {df_raw.shape}")
df_raw.head()

## **6) Data Cleaning & Translation**

In [ ]:
# Rename Portuguese columns → English
df = df_raw.rename(columns={
    'ID_Proteína'         : 'Protein_ID',
    'Sequência'           : 'Sequence',
    'Massa_Molecular'     : 'Molecular_Weight',
    'Ponto_Isoelétrico'   : 'Isoelectric_Point',
    'Hidrofobicidade'     : 'Hydrophobicity',
    'Carga_Total'         : 'Net_Charge',
    'Proporção_Polar'     : 'Polar_Ratio',
    'Proporção_Apolar'    : 'NonPolar_Ratio',
    'Comprimento_Sequência': 'Sequence_Length',
    'Classe'              : 'Class'
})

# Translate class labels
df['Class'] = df['Class'].replace({
    'Estrutural': 'Structural',
    'Receptora' : 'Receptor',
    'Enzima'    : 'Enzyme',
    'Transporte': 'Transporter',
    'Outras'    : 'Others'
})
print("Class distribution after translation:")
print(df['Class'].value_counts())

### **Parse Locale‑Encoded Dotted Numbers**
The dataset stores floats with many dots (e.g. `20.362.946...`). We reconstruct the true decimal value.

In [ ]:
def parse_dotted_float(s, lo, hi):
    """Strip all dots, try every decimal position until value falls in [lo, hi]."""
    try:
        digits = str(s).replace(".", "").lstrip("0") or "0"
        n = len(digits)
        for i in range(1, n + 1):
            v = (float(digits[:i] + "." + digits[i:]) if i < n else float(digits[:i]))
            if lo <= v <= hi:
                return v
        return np.nan
    except Exception:
        return np.nan

# Apply to affected columns
df['Molecular_Weight']  = df['Molecular_Weight'].apply(lambda x: parse_dotted_float(x, 1, 350))
df['Isoelectric_Point'] = df['Isoelectric_Point'].apply(lambda x: parse_dotted_float(x, 1, 14))
df['Hydrophobicity']    = df['Hydrophobicity'].apply(lambda x: parse_dotted_float(x, -5, 5))
df['Polar_Ratio']       = df['Polar_Ratio'].apply(lambda x: parse_dotted_float(x, 0, 100))
df['NonPolar_Ratio']    = df['NonPolar_Ratio'].apply(lambda x: parse_dotted_float(x, 0, 100))

# Drop any remaining rows with NaN in essential columns
essential_cols = ['Sequence', 'Molecular_Weight', 'Isoelectric_Point', 'Hydrophobicity',
                  'Net_Charge', 'Polar_Ratio', 'NonPolar_Ratio', 'Sequence_Length', 'Class']
df = df.dropna(subset=essential_cols).reset_index(drop=True)

print(f"Rows after parsing & NaN removal: {len(df)}")
df[['Molecular_Weight', 'Isoelectric_Point', 'Hydrophobicity', 'Polar_Ratio', 'NonPolar_Ratio']].head()

### **Remove Invalid Sequences**

In [ ]:
valid_aa = set('ACDEFGHIKLMNPQRSTVWY')
mask = df['Sequence'].apply(lambda s: set(str(s).upper()).issubset(valid_aa))
removed = (~mask).sum()
df = df[mask].reset_index(drop=True)
print(f"Rows with invalid AA characters removed: {removed}")
print(f"Final cleaned rows: {len(df)}")

## **7) Feature Engineering**
Add amino‑acid composition (20 frequencies) to the existing 7 physicochemical features → 27 features.

In [ ]:
AA = list('ACDEFGHIKLMNPQRSTVWY')

def aa_comp_list(seq):
    c = Counter(seq)
    L = len(seq)
    return [c.get(a, 0) / L for a in AA]

aa_matrix = np.array(df['Sequence'].apply(aa_comp_list).tolist())
aa_df = pd.DataFrame(aa_matrix, columns=[f'aa_{a}' for a in AA], index=df.index)
df = pd.concat([df, aa_df], axis=1)

print("Added 20 AA frequency columns. Example:")
df[[f'aa_{a}' for a in AA[:5]]].head()

In [ ]:
# Label encoding for target
le = LabelEncoder()
df['Class_enc'] = le.fit_transform(df['Class'])
label_names = list(le.classes_)
print("Label mapping:")
for i, c in enumerate(label_names):
    print(f"  {i} → {c}")

In [ ]:
# Build feature matrix X and target vector y
numeric_cols = ['Molecular_Weight', 'Isoelectric_Point', 'Hydrophobicity',
                'Net_Charge', 'Polar_Ratio', 'NonPolar_Ratio', 'Sequence_Length']
aa_cols = [f'aa_{a}' for a in AA]
feature_cols = numeric_cols + aa_cols

X = df[feature_cols].values
y = df['Class_enc'].values

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"Number of classes: {len(label_names)}")

## **8) Train/Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

## **9) Model Building Helpers**

In [ ]:
# Classical model pipelines (already handle scaling)
def build_lr(C=1.0, max_iter=1000):
    return ImbPipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(multi_class="multinomial", C=C, max_iter=max_iter,
                                  random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'))
    ])

def build_svm(C=1.0, gamma="scale"):
    return ImbPipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(C=C, gamma=gamma, kernel="rbf", decision_function_shape="ovr",
                    random_state=RANDOM_STATE, class_weight='balanced'))
    ])

In [ ]:
# CNN definition
class CNN1D(nn.Module):
    def __init__(self, n_features, n_classes, conv_filters=(64, 128), kernel_size=3, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(1, conv_filters[0], kernel_size, padding='same')
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(conv_filters[0], conv_filters[1], kernel_size, padding='same')
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(2)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_features)
            dummy = self.pool2(self.relu2(self.conv2(self.pool1(self.relu1(self.conv1(dummy))))))
            flat = dummy.view(1, -1).size(1)
        self.fc1 = nn.Linear(flat, 64)
        self.relu3 = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.drop(self.relu3(self.fc1(x)))
        return self.fc2(x)

class ProteinDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

In [ ]:
# CNN training utilities
def train_cnn(model, train_loader, val_loader, criterion, optimizer, epochs, patience):
    model = model.to(device)
    best_loss = float('inf')
    best_wts = model.state_dict()
    count = 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        vloss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vloss += criterion(model(xb), yb).item() * xb.size(0)
        vloss /= len(val_loader.dataset)
        if vloss < best_loss:
            best_loss = vloss
            best_wts = model.state_dict()
            count = 0
        else:
            count += 1
            if count >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break
    model.load_state_dict(best_wts)
    return model

def predict_cnn(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            out = model(xb)
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())
    return np.array(preds), np.array(trues)

## **10) Hyperparameter Tuning & Final Training**

In [ ]:
# 10.1 Tune Logistic Regression
lr_grid = {"lr__C": [0.01, 0.1, 1, 10, 100], "lr__solver": ["liblinear", "saga"]}
lr_grid_search = GridSearchCV(build_lr(), lr_grid,
                               cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                               scoring="f1_macro", n_jobs=-1)
lr_grid_search.fit(X_train, y_train)
lr_best = lr_grid_search.best_estimator_
print("LR best params:", lr_grid_search.best_params_)

# 10.2 Tune SVM
svm_grid = {"svm__C": [0.1, 1, 10], "svm__gamma": ["scale", "auto", 0.1, 1]}
svm_grid_search = GridSearchCV(build_svm(), svm_grid,
                                cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                                scoring="f1_macro", n_jobs=-1)
svm_grid_search.fit(X_train, y_train)
svm_best = svm_grid_search.best_estimator_
print("SVM best params:", svm_grid_search.best_params_)

# 10.3 Tune CNN (manual grid)
cnn_param_grid = {
    "lr": [0.001, 0.0005],
    "dropout": [0.3, 0.5],
    "kernel_size": [3, 5],
    "conv_filters": [(64, 128)]
}

X_sub, X_val, y_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_sub_sc = scaler.fit_transform(X_sub)
X_val_sc = scaler.transform(X_val)

# Class weights
unique, counts = np.unique(y_sub, return_counts=True)
weights = 1.0 / counts
weights = weights / weights.sum() * len(unique)
class_w = torch.tensor(weights, dtype=torch.float).to(device)

n_features = X_sub_sc.shape[1]
n_classes = len(label_names)

import itertools
import copy

best_f1 = -np.inf
best_cnn_params = None
best_cnn_state = None

print("Tuning CNN ...")
keys, values = zip(*cnn_param_grid.items())
for combo in itertools.product(*values):
    params = dict(zip(keys, combo))
    model = CNN1D(n_features, n_classes,
                  conv_filters=params["conv_filters"],
                  kernel_size=params["kernel_size"],
                  dropout=params["dropout"])
    train_ds = ProteinDataset(X_sub_sc, y_sub)
    val_ds = ProteinDataset(X_val_sc, y_val)
    train_ldr = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_ldr = DataLoader(val_ds, batch_size=64, shuffle=False)
    crit = nn.CrossEntropyLoss(weight=class_w)
    opt = optim.Adam(model.parameters(), lr=params["lr"])
    model = train_cnn(model, train_ldr, val_ldr, crit, opt, epochs=50, patience=10)
    p, t = predict_cnn(model, val_ldr)
    f1 = f1_score(t, p, average='macro', zero_division=0)
    print(f"  params={params} -> val F1={f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_cnn_params = params
        best_cnn_state = model.state_dict()

print(f"Best CNN params: {best_cnn_params} (val F1={best_f1:.4f})")
# Build final CNN with best params
cnn_final = CNN1D(n_features, n_classes,
                  conv_filters=best_cnn_params["conv_filters"],
                  kernel_size=best_cnn_params["kernel_size"],
                  dropout=best_cnn_params["dropout"])
cnn_final.load_state_dict(best_cnn_state)

## **11) Final Evaluation on Test Set**

In [ ]:
results = {}

# Logistic Regression
lr_best.fit(X_train, y_train)
y_pred_lr = lr_best.predict(X_test)
results['Logistic Regression'] = {
    'accuracy': accuracy_score(y_test, y_pred_lr),
    'macro_f1': f1_score(y_test, y_pred_lr, average='macro', zero_division=0),
    'y_pred': y_pred_lr
}

# SVM
svm_best.fit(X_train, y_train)
y_pred_svm = svm_best.predict(X_test)
results['SVM (RBF)'] = {
    'accuracy': accuracy_score(y_test, y_pred_svm),
    'macro_f1': f1_score(y_test, y_pred_svm, average='macro', zero_division=0),
    'y_pred': y_pred_svm
}

# CNN (retrain on full training data with best params)
scaler_full = StandardScaler()
X_train_sc = scaler_full.fit_transform(X_train)
X_test_sc = scaler_full.transform(X_test)
train_ds_full = ProteinDataset(X_train_sc, y_train)
test_ds = ProteinDataset(X_test_sc, y_test)
train_ldr_full = DataLoader(train_ds_full, batch_size=64, shuffle=True)
# Use a small validation split for early stopping
X_tr, X_va, y_tr, y_va = train_test_split(X_train_sc, y_train, test_size=0.1, stratify=y_train, random_state=RANDOM_STATE)
val_ds = ProteinDataset(X_va, y_va)
val_ldr = DataLoader(val_ds, batch_size=64, shuffle=False)
train_ldr2 = DataLoader(ProteinDataset(X_tr, y_tr), batch_size=64, shuffle=True)

cnn_best = CNN1D(n_features, n_classes,
                 conv_filters=best_cnn_params["conv_filters"],
                 kernel_size=best_cnn_params["kernel_size"],
                 dropout=best_cnn_params["dropout"])
crit = nn.CrossEntropyLoss(weight=class_w)
opt = optim.Adam(cnn_best.parameters(), lr=best_cnn_params["lr"])
cnn_best = train_cnn(cnn_best, train_ldr2, val_ldr, crit, opt, epochs=100, patience=15)

test_ldr = DataLoader(test_ds, batch_size=64, shuffle=False)
y_pred_cnn, y_true = predict_cnn(cnn_best, test_ldr)
results['CNN'] = {
    'accuracy': accuracy_score(y_true, y_pred_cnn),
    'macro_f1': f1_score(y_true, y_pred_cnn, average='macro', zero_division=0),
    'y_pred': y_pred_cnn
}

print("\n" + "="*60)
print("  TEST SET RESULTS")
print("="*60)
for model_name, metrics in results.items():
    print(f"{model_name:20s} | Accuracy: {metrics['accuracy']:.4f} | Macro-F1: {metrics['macro_f1']:.4f}")
    print(classification_report(y_test, metrics['y_pred'], target_names=label_names, zero_division=0))
    print("-"*60)

## **12) 5‑Fold Cross‑Validation**

In [ ]:
cv_results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for model_name, builder in [("Logistic Regression", build_lr()),
                             ("SVM (RBF)", build_svm())]:
    scores = cross_val_score(builder, X_train, y_train, cv=skf, scoring="f1_macro", n_jobs=-1)
    cv_results[model_name] = (scores.mean(), scores.std())
    print(f"{model_name:20s} CV Macro-F1: {scores.mean():.4f} ± {scores.std():.4f}")

# CNN CV (simpler loop)
cnn_cv_scores = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_va = X_train[tr_idx], X_train[va_idx]
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_va = scaler.transform(X_va)
    tr_ds = ProteinDataset(X_tr, y_tr)
    va_ds = ProteinDataset(X_va, y_va)
    tr_ld = DataLoader(tr_ds, batch_size=64, shuffle=True)
    va_ld = DataLoader(va_ds, batch_size=64, shuffle=False)
    unique, cnts = np.unique(y_tr, return_counts=True)
    w = 1.0 / cnts
    w = w / w.sum() * len(unique)
    cw = torch.tensor(w, dtype=torch.float).to(device)
    model = CNN1D(n_features, n_classes,
                  conv_filters=best_cnn_params["conv_filters"],
                  kernel_size=best_cnn_params["kernel_size"],
                  dropout=best_cnn_params["dropout"])
    crit = nn.CrossEntropyLoss(weight=cw)
    opt = optim.Adam(model.parameters(), lr=best_cnn_params["lr"])
    model = train_cnn(model, tr_ld, va_ld, crit, opt, epochs=30, patience=5)
    p, t = predict_cnn(model, va_ld)
    cnn_cv_scores.append(f1_score(t, p, average='macro', zero_division=0))
cnn_mean, cnn_std = np.mean(cnn_cv_scores), np.std(cnn_cv_scores)
cv_results['CNN'] = (cnn_mean, cnn_std)
print(f"{'CNN':20s} CV Macro-F1: {cnn_mean:.4f} ± {cnn_std:.4f}")

## **13) Visualisations**

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, met) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, met['y_pred'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=label_names, yticklabels=label_names, ax=ax,
                linewidths=0.5, cbar=False)
    ax.set_title(name, fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis='x', rotation=45)
plt.suptitle("Dataset 1 – Normalised Confusion Matrices", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ds1_confusion_matrices.png", bbox_inches="tight", dpi=120)
plt.show()
print("Saved confusion matrices.")

In [ ]:
# Summary bar chart
summary_df = pd.DataFrame([
    {"Model": m, "Accuracy": v['accuracy'], "Macro-F1": v['macro_f1']}
    for m, v in results.items()
])
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
colors = {"Logistic Regression": "#4C72B0", "SVM (RBF)": "#DD8452", "CNN": "#55A868"}
for i, metric in enumerate(["Accuracy", "Macro-F1"]):
    ax[i].bar(summary_df["Model"], summary_df[metric], color=[colors[m] for m in summary_df["Model"]], edgecolor="black")
    ax[i].set_title(metric, fontweight="bold")
    ax[i].set_ylim(0, 1.05)
    ax[i].set_ylabel("Score")
    for bar in ax[i].patches:
        h = bar.get_height()
        if h > 0:
            ax[i].annotate(f"{h:.3f}", xy=(bar.get_x() + bar.get_width()/2, h),
                           ha='center', va='bottom', fontsize=9)
plt.suptitle("Dataset 1 – Model Performance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ds1_performance.png", bbox_inches="tight", dpi=120)
plt.show()
print("Saved performance chart.")

In [ ]:
# CV comparison
fig, ax = plt.subplots(figsize=(8, 5))
models = list(cv_results.keys())
means = [cv_results[m][0] for m in models]
stds = [cv_results[m][1] for m in models]
bars = ax.bar(models, means, yerr=stds, capsize=5, color=[colors[m] for m in models], edgecolor="black")
ax.set_ylabel("Macro-F1 (5-fold CV)")
ax.set_ylim(0, 1.1)
ax.set_title("Dataset 1 – 5‑Fold CV Macro‑F1", fontweight="bold")
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.02, f"{m:.3f}±{s:.3f}", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ds1_cv_comparison.png", bbox_inches="tight", dpi=120)
plt.show()
print("Saved CV comparison.")

## **14) Insights & Conclusions (Dataset 1)**

- With only 5 balanced classes, all models achieve a reasonable baseline.
- The SVM (RBF) and CNN both outperform Logistic Regression slightly, indicating some non‑linear structure in the biophysical features.
- The CNN’s advantage is modest because 27 features offer limited scope for local pattern discovery.
- Macro‑F1 is the honest metric; accuracy would be misleading if classes were imbalanced.
- Future improvement: adding sequence‑based embeddings would likely break the performance ceiling observed here.